<h1 style="font-size:36px; line-height:1.1;">附录：时间感知知识图谱智能体</h1>

本笔记本包含**时间感知知识图谱智能体**指南的附录内容，为读者提供从原型到生产环境的完整实施方案。

在本附录中，您将找到更深入的*从原型到生产*章节，详细介绍如何将时间感知知识图谱系统从概念验证阶段扩展到高可用、高性能的生产环境。这些内容涵盖了系统架构、性能优化、成本控制和安全保障等关键方面。

## 附录内容概述
- **高容量图数据存储与检索**：处理百万或数十亿节点和边的策略，包括数据分区、索引优化和向量数据库选择
- **数据集管理与修剪**：TTL（生存时间）策略和智能剪枝技术，确保知识图谱的精简性和相关性
- **摄取管道并发实现**：将线性处理转换为可扩展的并行架构，提高系统吞吐量和可靠性
- **Token成本最小化**：缓存策略、灵活服务层级和API调用优化，降低OpenAI API使用成本
- **检索智能体扩展**：多跳查询的分布式处理，包括代理架构设计和并行子图提取技术
- **安全保障**：多层输出验证和审计日志记录机制，确保系统输出的质量和可靠性
- **提示优化**：角色设置、少样本提示、上下文管理和A/B测试，提高模型输出的质量和一致性

# A. 从原型到生产
---

## A.1. 高容量图数据的存储与检索

本节详细介绍如何处理大规模时间感知知识图谱的数据存储和检索挑战，包括数据架构设计、分区策略和索引优化技术。在处理包含数百万或数十亿节点和边的知识图谱时，合理的存储架构和检索优化对于系统性能至关重要。

### A.1.1. 数据量与模式复杂度

随着数据集扩展到数百万甚至数十亿节点和边，性能和可维护性变得至关重要。本节讨论如何设计可扩展的数据模式和分区策略。

随着数据集扩展到数百万甚至数十亿节点和边，性能和可维护性变得至关重要。这需要对模式设计和数据分区进行深思熟虑的方法：

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>面向增长和变化的模式设计</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      明确定义核心实体类型（如<code>Person</code>、<code>Organization</code>、<code>Event</code>）和关系。设计具有版本控制和灵活性的模式，使未来的模式演变能够以最小的停机时间进行。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>分片和分区</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      使用高基数字段（如时间戳或唯一实体ID）进行分区，以在数据量增长时保持查询性能。这对于时间感知数据尤为重要。例如：
    </p>
  
  ```sql  
  CREATE TABLE statements (
    statement_id UUID PRIMARY KEY,
    entity_id UUID NOT NULL,
    text TEXT NOT NULL,
    valid_from TIMESTAMP NOT NULL,
    valid_to TIMESTAMP,
    status VARCHAR(16) DEFAULT 'active',
    embedding VECTOR(1536),
    ...
  ) PARTITION BY RANGE (valid_from);
  ```
  </li>
</ol>

### A.1.2. 时间有效性与版本控制

在时间感知知识图谱中，每条语句都包含时间标记（如`valid_from`、`valid_to`）。本节介绍如何非破坏性地维护历史记录并优化时间访问性能，确保系统能够准确反映实体和关系随时间的变化，同时支持高效的历史查询。

在我们的时间感知知识图谱中，每个语句都包含时间标记（例如`valid_from`、`valid_to`）。

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>非破坏性地保留历史</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      避免删除或覆盖记录。相反，通过设置<code>status</code>（例如<code>inactive</code>）来标记过时的事实为非活动状态。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>优化时间访问</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      为时间字段（<code>valid_from</code>、<code>valid_to</code>）建立索引，以支持高效查询当前和历史状态。
    </p>
  </li>
</ol>


#### 示例：非破坏性更新

不是删除或覆盖记录，而是更新其状态并关闭其有效期窗口：

```sql
UPDATE statements
SET status = 'inactive', valid_to = '2025-03-15T00:00:00Z'
WHERE statement_id = '...' AND entity_id = '...';
```

### A.1.3. 索引与语义搜索

本节介绍如何创建高效的时间索引和向量索引，以支持复杂的时间查询和语义相似度搜索。通过合理的索引策略，可以显著提高查询性能，同时比较了不同向量数据库选项的优缺点，帮助选择最适合特定应用场景的解决方案。

##### 时间索引
为支持高效的时间查询，在<code>valid_from</code>和<code>valid_to</code>上创建B树索引。B树索引是一种自平衡的树数据结构，它保持数据排序以便以对数时间复杂度进行快速查找、范围查询和有序扫描。在处理时间序列数据时，B树索引能够显著提升查询性能，是关系数据库中最常用的索引类型之一。

```sql
CREATE INDEX ON statements (valid_from);
CREATE INDEX ON statements (valid_to);
```
##### 使用pgvector进行语义搜索
在PostgreSQL中存储向量嵌入（通过<code>pgvector</code>扩展）可以实现基于语义相似度的检索。pgvector扩展为PostgreSQL添加了向量数据类型和相似度搜索功能，使数据库能够直接处理和查询高维向量数据。实现语义搜索的步骤如下：
1. 存储表示文本语义含义的高维向量。这些向量可以使用专门的嵌入模型（如OpenAI的<code>text-embedding-3-small</code>和<code>text-embedding-3-large</code>）创建，这些模型能够捕捉文本的语义信息
2. 使用近似最近邻（ANN）算法进行大规模高效的相似度匹配，在保持合理精度的同时大幅提高查询速度

pgvector中有几种不同的索引选项，每种都有不同的用途。这些索引选项在pgvector的Github仓库上的README中有更详细的描述，以及深入的实现步骤。
| <div align="center">索引类型</div> | <div align="center">构建时间</div> | <div align="center">查询速度</div> | <div align="center">内存使用</div> | <div align="center">准确度</div> | <div align="center">推荐规模</div> | 说明 |
|-------------------------------------|--------------------------------------|----------------------------------------|-----------------------------------------|-----------------------------------|----------------------------------------------|-------|
| <div align="center">**flat**</div> | <div align="center">最小</div> | <div align="center">慢<br>(线性扫描)</div> | <div align="center">低</div> | <div align="center">100%<br>(精确)</div> | <div align="center">非常小<br>(&lt; 100 K向量)</div> | 无近似索引—扫描所有向量。最适合小型集合的精确召回 |
| <div align="center">**ivfflat**</div> | <div align="center">中等</div> | <div align="center">调优后快速</div> | <div align="center">中等</div> | <div align="center">高<br>(可调整)</div> | <div align="center">小型到中型<br>(100 K–200 M)</div> | 使用倒排文件索引。查询时参数控制权衡 |
| <div align="center">**ivfpq**</div> | <div align="center">高</div> | <div align="center">非常快</div> | <div align="center">低<br>(量化)</div> | <div align="center">略低于ivfflat</div> | <div align="center">中型到大型<br>(1 M–500 M)</div> | 结合倒排文件与乘积量化以降低内存使用 |
| <div align="center">**hnsw**</div> | <div align="center">最高</div> | <div align="center">最快<br>(尤其是大规模)</div> | <div align="center">高<br>(内存中)</div> | <div align="center">非常高</div> | <div align="center">大型到超大型<br>(100 M–数十亿+)</div> | 构建分层可导航图。适合对延迟敏感的大规模系统 |


##### 向量索引的调优参数

`ivfflat`
* `lists`: 分区数量（例如，100）
* `probes`: 查询时扫描的分区数量（例如，10-20），控制召回率与延迟的权衡

`ivfpq`
* `subvectors`: 量化的块数（例如，16）
* `bits`: 每块的位数（例如，8）
* `probes`: 与`ivfflat`相同

`hnsw`
* `M`: 每个节点的最大连接数（例如，16）
* `ef_construction`: 构建时动态候选列表大小（例如，200）
* `ef_search`: 查询时候选池（例如，64-128）

##### 最佳实践
- `flat`用于调试或小型数据集
- `ivfflat`当您需要可调准确度和良好速度时
- `ivfpq`当内存效率至关重要时
- `hnsw`当为大规模集合优化最低延迟时

##### 生态系统中的其他向量数据库选项

| 向量数据库 | 主要功能 | 优点 | 缺点 |
| ------------ | ------------------------------------------------------------ | ------------------------------------------- | --------------------------------------------------------------- |
| **Pinecone** | 完全托管，无服务器；支持HNSW和SPANN | 自动扩展，有SLA保障，易于集成 | 供应商锁定；大规模成本增加 |
| **Weaviate** | GraphQL API，内置编码和向量化模块 | 混合查询（元数据+向量），模块化 | 生产部署需要Kubernetes |
| **Milvus** | 支持GPU索引；IVF，HNSW，ANNOY | 大规模高性能，动态索引 | 操作复杂度高；独立系统 |
| **Qdrant** | 轻量级，实时更新，有效载荷过滤 | 简单设置，良好的混合查询支持 | 缺乏原生关系连接；集群中最终一致性 |
| **Vectara** | 托管服务，带语义排序和重排序 | 强大的相关性功能；易于集成 | 专有技术；索引控制有限 |

##### 选择正确的向量存储

| <div align="center">规模</div> | <div align="center">推荐</div> | 详细信息 |
|--------------------------------|------------------------------------------|---------|
| <div align="center">**小型到中型规模**<br>(少于100M向量)</div> | <div align="center">PostgreSQL + pgvector<br>带`ivfflat`索引</div> | 对于中等工作负载通常足够。推荐设置：`lists = 100–200`，`probes = 10–20`。 |
| <div align="center">**大规模**<br>(100M – 1B+向量)</div> | <div align="center">Milvus或Qdrant</div> | 适合高吞吐量工作负载，特别是需要GPU加速索引或亚毫秒级延迟时。 |
| <div align="center">**混合场景**</div> | <div align="center">PostgreSQL用于元数据<br>+ 专用向量数据库</div> | 使用PostgreSQL存储实体元数据，使用向量数据库（如Milvus、Qdrant）进行相似度搜索。使用CDC管道（如Debezium）同步嵌入。 |

有关更多详细信息，请查看[OpenAI关于向量数据库的指南](https://cookbook.openai.com/examples/vector_databases/readme)。

##### 持久磁盘存储和备份
对于某些情况，特别是那些需要高可用性或跨重启状态恢复的情况，可能值得将状态持久化到可靠的磁盘存储并实施备份策略。

如果持久性是一个问题，请考虑使用定期备份的持久磁盘或将状态同步到外部存储。虽然并非所有部署都需要，但在一致性和容错性很重要的环境中，它可以为防止数据丢失或操作中断提供宝贵的保障。

## A.2. 数据集管理与修剪

本节讨论如何建立有效的TTL（生存时间）策略和智能剪枝机制，以保持知识图谱的精简性和相关性，同时确保长期存储重要数据。随着知识图谱规模的增长，有效的数据管理策略对于维持系统性能和降低存储成本至关重要。

### A.2.1. TTL（生存时间）和归档策略

建立明确的策略来确定哪些事实应该无限期保留（如法律要求的记录），哪些可以在定义的时间段后归档（如一年以上的社交媒体声明）。

建立明确的策略来确定哪些事实应该无限期保留（例如，法律要求的监管记录）和哪些可以在定义的时期后归档（例如，一年以上的社交媒体声明）。

需要包括的关键实践：
<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>自动归档作业</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      设置定期查询记录的后台任务，例如<code>valid_to &lt; NOW() - INTERVAL 'X days'</code>，并将它们移动到归档表进行长期存储。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>源特定保留策略</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      按数据源或实体类型定制保留期限。例如，高权威来源（如政府出版物）可能需要比不太可靠的数据（如抓取的新闻标题或用户生成的内容）更长的保留期。
    </p>
  </li>
</ol>

### A.2.2. 相关性评分与智能剪枝

随着知识图谱的增长，许多事实的实用性会下降。本节介绍如何通过相关性评分和智能剪枝技术保持图谱的精简和高效。

随着知识图谱的增长，许多事实的实用性会下降。为了保持图谱的聚焦并最大化性能：
<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>索引相关性分数</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      引入数值<code>relevance_score</code>列（或多列），其中包含新近度、源可信度和生产查询频率等指标。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>自动剪枝逻辑</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      安排例行作业来剪枝或归档低于预定义相关性阈值的事实。
    </p>
  </li>
</ol>


#### 高级基于相关性的图简化

高效减少知识图谱的大小在扩展时很重要。[2024年的一项调查](https://arxiv.org/pdf/2402.03358)将技术分为**稀疏化**、**粗化**和**压缩**—所有这些都是为了在保留任务关键语义的同时缩小图。这些方法在大规模KG上提供了显著的运行时和内存收益。

示例实现模式：
<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>为每个三元组评分</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      计算复合<code>relevance_score</code>，例如：
    </p>
    <pre style="margin-top: 0.5em; margin-bottom: 0.5em; background-color: #f5f5f5; padding: 0.75em; border-radius: 5px;"><code>relevance_score = β1 * recency_score + β2 * source_trust_score + β3 * retrieval_count</code></pre>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      其中：
    </p>
    <ul style="margin-top: 0.5em; margin-bottom: 0.5em; padding-left: 1.2em;">
      <li><code>recency_score</code>：<code>valid_from</code>的指数衰减</li>
      <li><code>source_trust_score</code>：源域信任值</li>
      <li><code>retrieval_count</code>：生产查询频率</li>
    </ul>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>应用简化策略</strong><br />
    <ul style="margin-top: 0.5em; margin-bottom: 0.5em; padding-left: 1.2em;">
      <li><strong>稀疏化</strong>：基于中心性、谱相似度或嵌入保留等标准选择并保留最相关的边或节点</li>
      <li><strong>粗化</strong>：将低重要性或语义相似的节点组合成超级节点，并聚合它们的特征和连接</li>
      <li><strong>压缩</strong>：从头构建任务优化的迷你图</li>
    </ul>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>在影子模式下验证</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      在路由生产流量之前，记录并比较剪枝后与原始图的输出。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>定期重新评分</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      重新计算相关性（例如，每晚）以确保新的或频繁访问的事实回到顶部。
    </p>
  </li>
</ol>

## A.3. 摄取管道中的并发实现

本节介绍如何将线性处理管道转变为并发、可扩展的管道架构，通过批处理和并行工作来提高系统吞吐量和可靠性。并发实现是大规模数据处理系统的关键，能够显著提升数据处理速度并充分利用计算资源。

从原型到生产通常需要将线性处理管道转变为并发、可扩展的管道。不要按顺序处理文档（文档→分块→语句提取→实体提取→语句失效→实体解析），而是实现一个分阶段的管道，其中每个阶段可以独立扩展。

使用一系列专门的阶段设计您的管道，每个阶段都有自己的队列和工作池。这允许您独立扩展瓶颈并在不同负载下保持系统可靠性。

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>批处理分块</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      首先使用作业队列（如Redis或Amazon SQS）批量收集文档（例如，100–500个）。并行处理这些文档，将每个文档拆分为各自的块。分块阶段通常应优化I/O并行化，因为文档读取通常是瓶颈。然后，您可以使用批量插入操作将块及其各自的元数据存储在<code>chunk_store</code>表中，以最小化开销。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>语句和实体提取</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      批量（例如，50–100个）拉取块，并使用并行API请求将它们发送到您选择的LLM（例如，GPT-4.1-mini）。使用信号量或其他方法实现速率限制，以安全地保持在OpenAI的API限制内，同时最大化您的吞吐量。我们在关于<a href="https://cookbook.openai.com/examples/how_to_handle_rate_limits">如何处理速率限制</a>的指南中更详细地介绍了速率限制。一旦提取，您就可以将这些写入数据库中的相关表。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      然后，您可以类似地将我们刚刚提取的语句分组为批，并以类似的方式运行实体提取过程，然后再存储它们。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>语句失效</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      按关联的实体簇对提取的语句ID进行分组（例如，所有与特定实体（如"Acme Corp."）相关的语句）。并行地将每个簇发送到您的LLM（例如，GPT-4.1-mini），以评估哪些语句已过时或被取代。使用模型的输出来更新<code>statements</code>表中的<code>status</code>字段—例如，设置<code>status = 'inactive'</code>。并行化失效作业以提高性能，并考虑调度定期扫描以确保一致性。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>实体解析</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      提取新提取的实体提及批次，并使用模型的嵌入端点计算嵌入。将这些插入到<code>entity_registry</code>表中，为每个分配一个临时或规范的<code>entity_id</code>。使用<code>pgvector</code>执行近似最近邻（ANN）搜索来识别近似重复项或别名。然后，您可以使用解析后的规范ID更新<code>entities</code>表，确保下游任务引用统一的表示。
    </p>
  </li>
</ol>


### 批处理的优势
* 吞吐量 – 批处理减少了单个API调用和数据库事务的开销。

* 并行性 – 每个阶段可以水平扩展：您可以运行多个工作进程进行分块、提取、失效等，每个进程都从队列中读取。

* 背压和可靠性 – 如果一个阶段变慢（例如，数据突增期间的语句失效），上游阶段可以在队列中缓冲更多项目，直到容量释放。

## A.4. Token成本最小化

本节讨论如何通过提示缓存、灵活的服务层级和减少API调用频率来优化OpenAI API的使用成本，同时保持系统性能。在大规模AI系统中，API调用成本可能是主要开销之一，有效的成本优化策略对于系统的可持续运行至关重要。

### A.4.1. 提示缓存

通过缓存对相同或相似提示的响应，避免冗余的API调用，从而显著降低成本并提高响应速度。

通过记忆对脆弱子提示的响应，避免冗余的API调用。

实施策略：
- **缓存频繁查询**：例如，对相同语句重复的提示，如"从这个语句中提取实体"
- **使用哈希键**：使用语句文本的MD5哈希生成唯一的缓存键：`md5(statement_text)`
- **存储选项**：Redis用于可扩展持久性或内存LRU缓存用于简单性和速度
- **绕过API调用**：如果在缓存中找到语句，则跳过API调用

### A.4.2. 服务层级：Flex

利用OpenAI API的`service_tier=flex`参数启用部分完成，仅为生成的token计费，而不是提示token，从而大幅降低成本。

利用OpenAI Responses SDK中的`service_tier=flex`参数启用部分完成并降低成本。

API配置：
```json
{
  "model": "o4-mini",
  "prompt": "<your prompt>",
  "service_tier": "flex"
}
```

成本效益：
- 仅为生成的token计费，而不是提示token
- 对于短提取（如单句实体列表）可减少高达40%的成本

您可以在[Flex处理的API文档](https://platform.openai.com/docs/guides/flex-processing?api-mode=responses)中了解更多关于Flex处理的强大功能以及如何利用它。

### A.4.3. 减少"唠叨性"

在可能的情况下，用更高效的替代方案替换昂贵的文本生成调用，例如使用嵌入端点结合向量数据库进行相似度搜索。

在可能的情况下，用更高效的替代方案替换昂贵的文本生成调用。

替代方法：
- 使用嵌入端点（每个token更便宜）结合pgvector最近邻搜索
- 不是询问模型"哪个现有语句最相似？"，而是一次性计算嵌入并直接在Postgres中查询
- 这种方法对语义相似度任务特别有效

**优势：**
- 每次操作成本更低
- 查询响应时间更快
- 减少相似度搜索对API的依赖

## A.5. 检索智能体的扩展与生产化

本节介绍如何构建一个能够在大规模知识图谱上高效执行多跳查询的分布式检索系统，包括代理架构设计和并行子图提取技术。在生产环境中，检索系统需要处理高并发请求并在大规模数据上提供快速响应，分布式架构是实现这一目标的关键。

一旦您的图被填充，您需要一个机制来大规模回答多跳查询。这需要：

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>代理架构</strong><br />
    <ul style="margin-top: 0.5em; margin-bottom: 0.5em; padding-left: 1.2em;">
      <li><strong>控制器代理（前端）</strong>：接收用户问题（例如，"什么事件导致了Acme Corp.的IPO？"），然后将其分解为子问题或遍历步骤。</li>
      <li><strong>遍历工作代理</strong>：每个工作者可以执行局部图遍历（例如，"查找Acme Corp.在2020–2025年间具有EventType = Acquisition的所有事实"），可能并行在图的不同分区上执行。</li>
    </ul>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>并行子图提取</strong><br />
    <ul style="margin-top: 0.5em; margin-bottom: 0.5em; padding-left: 1.2em;">
      <li>按实体ID哈希对图进行分区（例如，模16）。对于给定查询，识别哪些分区可能包含相关边，然后将遍历任务并行分配给每个工作者。</li>
      <li>工作者返回部分子图（节点+边），控制器代理合并它们。</li>
    </ul>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>链式LLM推理</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      对于多跳问题，控制器可以用部分子图提示模型（如GPT-4.1）并询问"我应该遍历哪个下一条边？"这允许动态、上下文感知的遍历，而不是盲目的广度优先搜索。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>缓存和记忆化</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      对于经常问到的查询或子图模式，将结果缓存（例如，在Redis或Postgres物化视图中），TTL等于事实的<code>valid_to</code>日期，以便后续请求命中缓存而不是重新遍历。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>负载平衡和自动扩展</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      在Kubernetes集群中部署遍历工作代理，使用水平Pod自动缩放器。使用CPU和内存指标（以及平均队列长度）在峰值使用期间进行扩展。
    </p>
  </li>
</ol>


## A.6. 安全保障

本节讨论如何实现多层输出验证和审计日志记录机制，以确保系统输出的质量和可靠性，同时提供完整的操作追踪能力。在生产环境中，安全保障机制对于维护系统的稳定性、可靠性和可审计性至关重要，能够帮助及时发现和解决问题。

### A.6.1. 多层输出验证

运行轻量级验证管道，确保输出符合预期格式和质量标准，包括日期格式检查、实体类型验证和一致性检查。

运行轻量级验证管道，确保输出如预期。可包含在此中的一些示例：
* 检查日期是否符合<code>ISO-8601</code>格式
* 验证实体类型是否匹配您的受控词汇表（例如，如果模型输出意外标签，则标记为手动审查）
* 部署"健全性检查"函数调用到更小、更便宜的模型，以验证输出的一致性（例如，"这个语句是否正确解析为事实？是/否。"）

### A.6.2. 审计日志记录与监控

实现结构化日志记录和全面监控，跟踪系统性能、数据质量和业务指标，以便及时发现和解决问题。

- 实现具有可配置详细程度级别的结构化日志记录（例如，调试、信息、警告、错误）
- 存储输入预处理步骤、中间输出和最终结果，并提供完整的跟踪，如[OpenAI的跟踪](https://platform.openai.com/traces)所提供的
- 跟踪token吞吐量、延迟和错误率
- 在可能的情况下监控数据质量指标，如文档或语句覆盖率、时间分辨率率等
- 测量业务相关指标，如用户数量、平均消息量和用户满意度

## A.7. 提示优化

本节介绍如何通过角色设置、少样本提示、上下文管理和A/B测试来优化提示效果，提高模型输出的质量和一致性。在AI系统中，提示设计是影响模型性能的关键因素，良好的提示策略能够显著提升系统的准确性和效率。

<ol style="margin-left: 1em; line-height: 1.6; padding-left: 0.5em;">
  <li style="margin-bottom: 1.2em;">
    <strong>角色设置</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      向模型介绍角色是提高性能的有效方法。通过为模型设定明确的角色和专业背景，可以引导模型以更专业、更一致的方式回应查询。一旦确定了您正在开发提示的组件的专业领域，您可以在系统提示中创建一个详细的角色设定，帮助塑造模型的行为。我们在规划器模型中使用了这一点，创建了如下系统提示：
    </p>
    <pre style="margin-top: 0.5em; margin-bottom: 0.5em; background-color: #f5f5f5; padding: 0.75em; border-radius: 5px;"><code>initial_planner_system_prompt = (
    "您是全球领先金融公司ABC Incorporated的资深顾问，该公司是世界上最大的金融公司之一。"
    "凭借您在公司的长期卓越工作，各股票研究团队经常向您寻求关于他们正在执行的研究任务的指导。"
    "您的专业知识尤其强大，特别是在ABC Incorporated专有的财报电话会议记录知识库方面。"
    "该知识库包含从各公司财报电话会议记录中提取的详细信息，并标注了这些陈述的有效时间。"
    "您是指导团队如何使用这一知识图谱回答他们研究查询的专家。\n"
)</code></pre>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      角色提示可以根据具体任务需求更加复杂和具体，上述示例展示了在金融领域中角色设定的基本模式。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>少样本提示和思维链</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      对于提取相关任务，如语句提取，简洁的少样本提示（提供2–5个示例）通常会比零样本提示提供更高的精度，尽管成本会略有增加。通过向模型展示期望的输入输出模式，可以显著提高模型的理解和执行能力。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      对于需要复杂推理的任务，如时间协调，思维链方法（引导模型通过逻辑步骤进行推理）更为合适。这种方法鼓励模型展示其推理过程，提高结果的可解释性和准确性。例如：
    </p>
    <pre style="margin-top: 0.5em; margin-bottom: 0.5em; background-color: #f5f5f5; padding: 0.75em; border-radius: 5px;"><code>示例1: [旧事实], [新事实] → 使失效
示例2: [旧事实], [新事实] → 共存
现在: [旧事实], [新事实] →</code></pre>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>动态提示和上下文管理</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      您还可以依靠其他LLM或更结构化的方法来修剪和准备将动态传递给提示的材料。我们在构建检索器工具时看到了一个例子，其中<code>timeline_generation</code>工具在将检索到的材料传递回中央协调器之前对其进行排序，确保信息以最有意义的方式呈现。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      在运行中期清理上下文或压缩上下文的步骤对于运行时间较长的查询也可以非常有效。这可以通过总结已完成部分的关键信息来实现，释放上下文窗口空间以处理新的信息。
    </p>
  </li>

  <li style="margin-bottom: 1.2em;">
    <strong>模板库和A/B测试</strong><br />
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      在版本控制目录中维护一组提示模板（例如，<code>prompts/statement_extraction.json</code>、<code>prompts/entity_extraction.json</code>），以便您可以审计过去的更改并在必要时回滚。您可以利用OpenAI的可重用提示功能来实现这一点。在OpenAI仪表板中，您可以开发<a href="https://platform.openai.com/docs/guides/text#reusable-prompts">可重用提示</a>以在API请求中使用，这使您能够构建和评估提示，部署更新版本，而无需更改代码。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      实现自动化A/B测试流程，通过定期从处理管道中采样提取的事实，使用不同的提示模板重新运行它们，并比较性能分数。这可以帮助您持续优化提示效果，选择最适合特定任务的提示策略。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      跟踪关键性能指标（KPI），如提取延迟、错误率和事实失效准确性等。建立完善的监控系统，及时发现性能异常。
    </p>
    <p style="margin-top: 0.5em; margin-bottom: 0.5em;">
      设置自动警报机制，如果任何关键指标超出预设阈值（例如，失效准确性低于90%），则触发警报并自动回滚到之前验证过的提示版本，确保系统的稳定运行。
    </p>
  </li>
</ol>
